# Micro Mission 6 Lab: Reading a Markup off Demand
## ECON 2316, Chapters 11 and 12
### About 45 minutes, including the write-up. Nothing to install.

**What you do.** You run a demand model on cereal data and read results off the output. Each
result is one number. Then you explain what the numbers mean with the Chapter 11 and 12 formulas.
**You do not write code.** Every step's code is given and hidden; you press Run and read.

**The data.** `pyblp`, a library economists use to estimate demand, ships a teaching dataset
modelled on Aviv Nevo's (2000) study of ready-to-eat cereal: 24 brands, 5 firms, 47 cities,
2 quarters. **The numbers are made up** so that they look like real supermarket data. So every
result in this lab is *what the model says on teaching data*. It is not a fact about real cereal
companies, and your write-up should say it that way.

| Part | Question | Chapter formula |
|---|---|---|
| 1 | How fast does demand fall when price rises? | the price coefficient |
| 2 | How elastic is each brand's demand? | $MR = P(1 + 1/\varepsilon_D)$, Heads Up 11.3 |
| 3 | What markup does that elasticity imply? | $L = (P - MC)/P = -1/\varepsilon_D$, §11.5 |
| 4 | Would one price fit every city? | $L_i = -1/\varepsilon_{D,i}$, Calculus Corner 12.1 |
| 5 | Write-up: three short answers | all of the above |

**Before each estimate you predict the result** (multiple choice). Being wrong costs nothing;
your one-sentence explanation of what happened is what counts.

> ### How to use this notebook (no coding needed)
>
> - **Save your own copy first:** *File → Save a copy in Drive*.
> - **Run everything:** *Runtime → Run all*. The first cell installs `pyblp` (about a minute); everything else takes a few seconds. If Colab warns that the notebook was not authored by Google, choose *Run anyway*.
> - Each grey cell shows its title and its output. *Show code* is there if you are curious; you never need it.
> - Keep a note of your four predictions and the five numbers. Your Canvas post asks for them.

In [ ]:
# @title Setup: install pyblp and load the libraries { display-mode: "form" }
# GIVEN: run as-is. pyblp is Conlon and Gortmaker's demand-estimation library.
import importlib.util, subprocess, sys, warnings
if importlib.util.find_spec("pyblp") is None:
    print("Installing pyblp (about a minute the first time) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyblp>=1.2,<2"], check=True)

import numpy as np
import pandas as pd
import pyblp
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")          # pyblp prints a few harmless numerical notes
pyblp.options.verbose = False              # keep the output short
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False})

def _cmp3(a, b, band=0.05):
    # 0 / 1 / 2 = a is below / within 5% of / above b (used to score a prediction)
    return 1 if abs(a - b) <= band * abs(b) else (0 if a < b else 2)

print(f"pyblp {pyblp.__version__} | numpy {np.__version__} | pandas {pd.__version__} | Python {sys.version.split()[0]}")
print("Setup complete.")

---
## The data (3 min)

One row is one brand in one market, and a **market** is one city in one quarter. For each row the
data give the price per serving and the brand's **market share**: its servings sold divided by all
the servings people in that city could have eaten (one per person per day). Whatever is left over
is the **outside option**, no cereal at all. Brands and firms are numbers, not names: the data are
made up.

In [ ]:
# @title Load the cereal data and look at it { display-mode: "form" }
# GIVEN: run as-is. pyblp's teaching copy of the Nevo (2000) cereal data (synthetic numbers).
df = pd.read_csv(pyblp.data.NEVO_PRODUCTS_LOCATION)      # df is read here and never changed later

print(f"{len(df):,} rows: one row per brand per market")
print(f"{df.city_ids.nunique()} cities x {df.quarter.nunique()} quarters = {df.market_ids.nunique()} markets")
print(f"{df.product_ids.nunique()} brands, owned by {df.firm_ids.nunique()} firms")
print(f"Median price per serving: ${df.prices.median():.3f}.  Median market share: {df.shares.median():.1%}.")
print(f"In the median market, {1 - df.groupby('market_ids').shares.sum().median():.0%} of possible servings are 'no cereal'.")

print("\nThe first six rows:")
display(df[["city_ids", "quarter", "product_ids", "firm_ids", "prices", "shares"]].head(6)
        .rename(columns={"city_ids": "city", "product_ids": "brand", "firm_ids": "firm",
                         "prices": "price ($/serving)", "shares": "market share"}))
owners = df.groupby("firm_ids").product_ids.nunique().rename("brands owned").to_frame()
owners.index = [f"Firm {i}" for i in owners.index]
print("Who owns how many brands:")
display(owners)

---
## Part 1: How fast does demand fall when price rises? (8 min)

The model is a **logit**: each shopper picks the one cereal, or no cereal, that gives the highest
utility. A brand's average utility in a market is

$$\delta_{jt} = \alpha\, p_{jt} + (\text{brand effect})_j + \xi_{jt}$$

- $p_{jt}$ is the price per serving of brand $j$ in market $t$.
- $\alpha$ is the **price coefficient**, the slope we want: how much utility falls per dollar of price.
- The **brand effect** is one constant per brand (loyalty to the brand). With it in the model, $\alpha$ is learned from the *same* brand's price moving across cities and quarters.
- $\xi_{jt}$ is what shoppers see and the data do not: a store display, a local ad campaign.

**Why the step uses instruments.** Firms charge more where $\xi$ is high, where the brand is popular
for reasons the data do not show. So high prices and high demand turn up together, and a plain
regression understates how much price hurts demand. The fix is an **instrument**: a variable
that moves price for cost reasons and has nothing to do with $\xi$. The data come with 20 of them,
and the step names all 20 explicitly. You do not need the econometrics for the write-up; you
need the sign and the size of $\alpha$.

> **Predict before you run.** What sign will the price coefficient $\alpha$ have?
>
> - Negative: a higher price lowers utility
> - About zero: price does not matter much for cereal
> - Positive: a higher price signals a better cereal
>
> Write down your choice before you run the next cell.

In [ ]:
# @title Part 1: estimate demand (logit, instrumented price, brand fixed effects) { display-mode: "form" }
# GIVEN: run as-is.
# The instrument set is written out here, and the step builds its own copy of the columns it needs,
# so it gives the same answer however often, and in whatever order, the steps are run.
IV_COLS = [f"demand_instruments{k}" for k in range(20)]
COLS = ["market_ids", "city_ids", "quarter", "firm_ids", "product_ids", "prices", "shares"] + IV_COLS
demand = pyblp.Problem(pyblp.Formulation("0 + prices", absorb="C(product_ids)"), df[COLS])
logit = demand.solve(method="1s")            # one-step GMM with these instruments = plain IV (2SLS)

alpha = float(logit.beta[0, 0])
alpha_se = float(logit.beta_se[0, 0])
print(f"Price coefficient alpha: {alpha:.2f}   (standard error {alpha_se:.2f})")
print(f"So a 1-cent rise in a brand's price lowers its average utility by {abs(alpha) * 0.01:.2f}.")
print(f"At the median price (${df.prices.median():.3f} per serving), alpha x price = {alpha * df.prices.median():.2f}.")

> **Read it off your output.** The price coefficient $\alpha$, **with its sign**, to one decimal.

> **Chapter formula.** In the logit, a brand's own-price elasticity is $\varepsilon_{jt} = \alpha\, p_{jt}\,(1 - s_{jt})$, where $s_{jt}$ is its market share. A higher price or a smaller share makes demand more elastic. Part 2 computes it for every row.

> **Use the formula.** Use your $\alpha$ and the formula: a brand priced at 20 cents (0.20 dollars) per serving with a 2% share ($s = 0.02$). What is its own-price elasticity? Two decimals, with the sign.

---
## Part 2: How elastic is each brand's demand? (8 min)

The next step computes $\varepsilon_{jt}$ for all 2,256 rows: every brand in every market. Recall
§11.3: marginal revenue is $MR = P\,(1 + 1/\varepsilon_D)$. Heads Up 11.3 says what that implies
for where a profit-maximizing firm can be on its demand curve.

> **Predict before you run.** For the typical (median) brand and market, will demand be inelastic, about unit-elastic, or elastic?
>
> - Inelastic: between 0 and −1
> - About unit-elastic: close to −1
> - Elastic: below −1 (for example −2 or −4)
>
> Write down your choice before you run the next cell.

In [ ]:
# @title Part 2: the own-price elasticity of every brand in every market { display-mode: "form" }
# GIVEN: run as-is. Uses Part 1's estimate (logit).
E = logit.compute_elasticities()               # one matrix per market: how each share responds to each price
own = logit.extract_diagonals(E).ravel()       # the diagonal = each brand's OWN-price elasticity
eps_median = float(np.median(own))

print(f"{len(own):,} own-price elasticities (24 brands x 94 markets)")
print(f"Median: {eps_median:.2f}")
print(f"Middle half of the rows: {np.percentile(own, 25):.2f} to {np.percentile(own, 75):.2f}")
print(f"Most elastic row: {own.min():.2f}.  Least elastic row: {own.max():.2f}")
print(f"Rows in the elastic part of demand (|elasticity| > 1): {(np.abs(own) > 1).mean():.0%}")

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(own, bins=40, color="#2a78d6", edgecolor="white")
ax.set_ylim(0, ax.get_ylim()[1] * 1.3)          # headroom for the two labels
ax.axvline(-1, color="#b42318", ls="--", lw=1.5)
ax.text(-1.08, ax.get_ylim()[1] * 0.92, "unit-elastic\n(ε = −1)", ha="right", va="top", color="#b42318", fontsize=9)
ax.axvline(eps_median, color="#1a1a19", lw=1.2)
ax.text(eps_median + 0.08, ax.get_ylim()[1] * 0.92, f"median {eps_median:.2f}", ha="left", va="top", fontsize=9)
ax.set_xlim(-7.5, 0)
ax.set_xlabel("own-price elasticity of demand, ε")
ax.set_ylabel("number of rows")
ax.set_title("Own-price elasticities, 24 brands x 94 markets (teaching data)", fontsize=10)
plt.tight_layout()
plt.show()

> **Read it off your output.** The **median** own-price elasticity, two decimals, with its sign.

> **Chapter formula.** §11.3: $MR = P\,(1 + 1/\varepsilon_D)$. Heads Up 11.3: where $|\varepsilon_D| < 1$, $MR < 0$, so a profit-maximizing firm never prices on the inelastic part of its demand curve.

> **Use the formula.** Suppose one brand had $\varepsilon = -0.8$. Its marginal revenue $MR = P(1 + 1/\varepsilon)$ would be …
>
> - positive
> - zero
> - negative

---
## Part 3: From elasticity to markup (12 min)

§11.5: a profit-maximizing firm sets $MR = MC$. With $MR = P(1 + 1/\varepsilon_D)$ this rearranges to

$$L = \frac{P - MC}{P} = -\frac{1}{\varepsilon_D}.$$

Read it backwards: **if you know the elasticity, you know the markup**, without ever seeing costs.
That is how economists measure market power from demand data. The formula is for a firm that sells
**one** product. Step 3a uses it as if every brand were its own firm.

In [ ]:
# @title Part 3a: the single-product Lerner index, L = −1/ε { display-mode: "form" }
# GIVEN: run as-is. Uses Part 2's elasticities (own).
lerner_single = -1 / own
L_single = float(np.median(lerner_single))
print(f"Median single-product Lerner index: {L_single:.2f}")
print(f"Middle half of the rows: {np.percentile(lerner_single, 25):.2f} to {np.percentile(lerner_single, 75):.2f}")
print(f"So at the median, the gap between price and marginal cost is {L_single:.0%} of the price.")

> **Read it off your output.** The **median single-product Lerner index** $-1/\varepsilon$, two decimals.

> **Use the formula.** Walkthrough 11.2 put Spotify Premium's elasticity at $\varepsilon_D = -3.33$. What Lerner index does the inverse-elasticity rule give? Two decimals.

### 3b: the firms own several brands

In these data 5 firms own the 24 brands: Firms 1 and 2 own nine each, Firm 6 owns one. When a
firm sets prices it looks at **all of its brands together**. pyblp's `compute_costs` works
backwards from each firm's pricing first-order conditions: given the demand you estimated and who
owns what, which marginal cost would make today's prices the profit-maximizing ones? From that
$MC$ the step computes the markup $(P - MC)/P$.

> **Predict before you run.** Compared with the single-product Lerner $-1/\varepsilon$ from 3a, will the median markup be smaller, about the same, or larger when each firm prices all of its brands together?
>
> - Smaller
> - About the same (within 5%)
> - Larger
>
> Write down your choice before you run the next cell.

In [ ]:
# @title Part 3b: markups when each firm prices all its brands together { display-mode: "form" }
# GIVEN: run as-is. Uses Part 1's estimate (logit) and Part 2's elasticities (own).
price = df.prices.values
mc = logit.compute_costs().ravel()                  # firm_ids in the data say who owns which brand
markup = (price - mc) / price
L_portfolio = float(np.median(markup))

single = -1 / own                                    # recomputed here, so this step does not depend on 3a
mc_alone = logit.compute_costs(firm_ids=np.arange(len(df))).ravel()   # pretend every brand is its own firm
same = np.allclose((price - mc_alone) / price, single)

print(f"Median markup (P - MC)/P with the real owners: {L_portfolio:.2f}")
print(f"Median single-product Lerner -1/elasticity:    {float(np.median(single)):.2f}")
print(f"Check: with every brand as its own firm, compute_costs gives back -1/elasticity exactly: {same}")

firm_table = (pd.DataFrame({"firm": df.firm_ids, "brand": df.product_ids, "single": single, "portfolio": markup})
              .groupby("firm").agg(brands=("brand", "nunique"), single=("single", "median"), portfolio=("portfolio", "median")))
firm_table.index = [f"Firm {i}" for i in firm_table.index]
firm_table.columns = ["brands owned", "single-product L = -1/elasticity (median)", "markup with real owners (median)"]
print("\nBy firm:")
display(firm_table)

> **Read it off your output.** The **median markup with the real owners**, two decimals.

---
## Part 4: One price for every city? (6 min)

Chapter 12 (§12.4, Calculus Corner 12.1): if a firm can tell segments apart and stop resale between
them, it sets each segment's price by that segment's own Lerner rule. Here the segments are the 47
cities. **Nothing is re-estimated**: the step takes the Part 2 elasticities and groups them by city.

> **Predict before you run.** Suppose a cereal firm could set a different price in each city, and nobody could resell between cities. Which city would pay the higher markup?
>
> - The city where demand is more elastic
> - The city where demand is less elastic
> - Neither: marginal cost is the same everywhere, so the markup is the same
>
> Write down your choice before you run the next cell.

In [ ]:
# @title Part 4: elasticities and implied markups, city by city { display-mode: "form" }
# GIVEN: run as-is. Uses Part 2's elasticities (own); no new estimation.
city = (pd.DataFrame({"city": df.city_ids, "elasticity": own})
        .groupby("city").elasticity.median()
        .sort_values(ascending=False)                # least elastic first
        .to_frame("median elasticity"))
city["implied L = -1/elasticity"] = -1 / city["median elasticity"]
city.index = [f"City {c}" for c in city.index]
city_L = city["implied L = -1/elasticity"]

print(f"{len(city)} cities. Median elasticity runs from {city.iloc[0, 0]:.2f} ({city.index[0]}, least elastic) "
      f"to {city.iloc[-1, 0]:.2f} ({city.index[-1]}, most elastic).")
print("The five least elastic cities (top) and the five most elastic (bottom):")
display(pd.concat([city.head(5), city.tail(5)]))

> **Read it off your output.** The implied Lerner index in the **least elastic city** (the first row), two decimals.

> **Chapter formula.** Calculus Corner 12.1: with no resale between segments, each segment $i$ gets $L_i = (P_i - MC)/P_i = -1/\varepsilon_{D,i}$. Same $MC$ everywhere, different $\varepsilon_{D,i}$, different markup.

---
## Your five numbers (1 min)

The next step collects the five numbers you read off. They go at the top of your Canvas post.

In [ ]:
# @title Your five numbers, for your Canvas post { display-mode: "form" }
# GIVEN: run as-is.
summary = pd.DataFrame({
    "Part": ["1", "2", "3a", "3b", "4"],
    "What": ["price coefficient alpha", "median own-price elasticity", "median single-product Lerner -1/elasticity",
             "median markup with the real owners", "implied Lerner, least elastic city"],
    "Value": [f"{alpha:.2f}", f"{eps_median:.2f}", f"{float(np.median(-1 / own)):.2f}", f"{L_portfolio:.2f}",
              f"{float(city_L.iloc[0]):.2f} ({city_L.index[0]})"],
}).set_index("Part")
display(summary)
print("All five are what the model says on pyblp's synthetic teaching data.")

---
## Part 5: Write-up (15 min)

Answer the three questions, **about 80 words each (about 250 in all)**. Each answer must use the
formula named in it and at least one of your numbers. Write results as what the model says on
teaching data (for example "in the model, the median markup is …"), not as facts about real cereal
companies.

1. **Elastic or not ($MR = P(1 + 1/\varepsilon_D)$, Heads Up 11.3).** Is your median own-price elasticity where a profit-maximizing firm could be pricing? Explain with the MR formula, and say what a median of −0.6 would have told you.
2. **Owning substitutes ($L = -1/\varepsilon_D$, §11.5).** Your single-product Lerner index and the markup with the real owners differ. Which is larger, and why does owning several brands change how a firm prices each one? Use the firm table (compare Firm 6 with Firm 1). Chapter 13 starts from here.
3. **One price for every city? ($L_i = -1/\varepsilon_{D,i}$, Calculus Corner 12.1).** If a cereal firm could set a different price in each city, which city would pay the higher markup, and what would the firm need for that to work (§12.1)?

> **Answer Write-up** in your Canvas post.

---
## Submit

Post in the **Micro Mission 6** discussion on Canvas:

- your five numbers (the table above),
- your four predictions, each with one sentence on what happened and why,
- your three write-up answers (about 250 words in all).

Then reply to one classmate (two or three sentences). Nothing was installed on your computer.